# 1. Setup & Installation

In [ ]:
# Install required libraries (run once)
!pip install langchain langchain-openai faiss-cpu openai

# 2. Imports & API Setup

In [ ]:
import os

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

from langchain_openai import ChatOpenAI

# 3. Basic LLM Call

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

response = llm.invoke("Explain LangChain in simple terms")
print(response.content)

# 4. Prompt Templates

In [ ]:
from langchain.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

prompt = template.format(topic="LangChain")
print(prompt)

# 5. Chains

In [ ]:
from langchain.chains import LLMChain

chain = LLMChain(llm=llm, prompt=template)

result = chain.run("Vector Databases")
print(result)

# 6. Memory (Conversation)

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory
)

print(conversation.predict(input="Hi, I am Tejas"))
print(conversation.predict(input="What is my name?"))

# 7. Tools + Agent

In [ ]:
from langchain.agents import initialize_agent, Tool

# Simple calculator tool
def calculator_tool(input):
    return str(eval(input))

tools = [
    Tool(
        name="Calculator",
        func=calculator_tool,
        description="Performs math calculations"
    )
]

agent = initialize_agent(
    tools,
    llm,
    agent="zero-shot-react-description",
    verbose=True
)

agent.run("What is 45 * 12?")

# 8. Document Loader

In [ ]:
from langchain.document_loaders import TextLoader

# Create sample file first
with open("sample.txt", "w") as f:
    f.write("LangChain helps in building LLM applications.")

loader = TextLoader("sample.txt")
documents = loader.load()

documents

# 9. Embeddings + Vector Store (FAISS)

In [ ]:
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

embedding = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(documents, embedding)

query = "What is LangChain?"

docs = vectorstore.similarity_search(query)

for doc in docs:
    print(doc.page_content)

# 10. Simple RAG Pipeline

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever()
)

response = qa_chain.run("Explain LangChain")
print(response)

# 11. Modular Function Version

In [ ]:
def create_llm():
    return ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def run_prompt(topic):
    template = PromptTemplate(
        input_variables=["topic"],
        template="Explain {topic} clearly"
    )
    chain = LLMChain(llm=create_llm(), prompt=template)
    return chain.run(topic)

print(run_prompt("AI Agents"))